# ✈️ Air India RAG Chatbot

A Retrieval-Augmented Generation (RAG) chatbot built with:
- **LangChain** (v0.2+)
- **ChromaDB** as vector store
- **HuggingFace** embeddings (free, no AWS needed)
- **Groq** (LLaMA Instant) or **OpenAI** as LLM
- **Gradio** for the interactive UI

---
### 📋 Steps:
1. Install dependencies
2. Upload your Air India PDF(s)
3. Build the vector store
4. Choose your LLM (Groq or OpenAI)
5. Launch the chatbot app

## 🔧 Step 1: Install Dependencies

In [2]:
!pip install -q \
    langchain\
    langchain-community \
    langchain-chroma\
    langchain-groq\
    langchain-openai\
    langchain-huggingface\
    chromadb \
    sentence-transformers \
    pypdf \
    gradio \
    tiktoken

print("✅ All dependencies installed!")

✅ All dependencies installed!


## 🔑 Step 2: Load API Keys from Colab Secrets

Go to the 🔑 **Secrets** panel (left sidebar) and add:
- `GROQ_API_KEY` → your Groq API key (required for Groq LLM)
- `OPENAI_API_KEY` → your OpenAI API key (optional, if using OpenAI)

In [3]:
import os
from google.colab import userdata

# Load Groq API Key (required)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    print("✅ GROQ_API_KEY loaded successfully")
except Exception:
    print("⚠️  GROQ_API_KEY not found in Colab Secrets. Groq LLM will not be available.")
    GROQ_API_KEY = None

# Load OpenAI API Key (optional)
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print("✅ OPENAI_API_KEY loaded successfully")
except Exception:
    print("ℹ️  OPENAI_API_KEY not found — OpenAI LLM will not be available.")
    OPENAI_API_KEY = None

✅ GROQ_API_KEY loaded successfully
ℹ️  OPENAI_API_KEY not found — OpenAI LLM will not be available.


## 📄 Step 3: Upload Your PDF Files

Upload one or more Air India related PDF files when prompted below.

In [4]:
import os
from google.colab import files

PDF_DIR = "/content/air_india_docs"
os.makedirs(PDF_DIR, exist_ok=True)

print("📂 Please upload your Air India PDF file(s)...")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest_path = os.path.join(PDF_DIR, filename)
    with open(dest_path, 'wb') as f:
        f.write(content)
    print(f"✅ Saved: {filename} ({len(content):,} bytes)")

pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]
print(f"\n📚 Total PDFs ready: {len(pdf_files)}")
for pdf in pdf_files:
    print(f"   • {pdf}")

📂 Please upload your Air India PDF file(s)...


Saving Aiesl Employees service regulation.pdf to Aiesl Employees service regulation.pdf
Saving Air India Fact Sheet.pdf to Air India Fact Sheet.pdf
Saving Domestic Routes Feb 2025.pdf to Domestic Routes Feb 2025.pdf
Saving International Routes Feb 2025.pdf to International Routes Feb 2025.pdf
Saving List of Major Air India Disasters _ Crashes, Death Toll, Tata Group, History, & Accidents _ Britannica.pdf to List of Major Air India Disasters _ Crashes, Death Toll, Tata Group, History, & Accidents _ Britannica.pdf
✅ Saved: Aiesl Employees service regulation.pdf (3,328,928 bytes)
✅ Saved: Air India Fact Sheet.pdf (430,799 bytes)
✅ Saved: Domestic Routes Feb 2025.pdf (770,446 bytes)
✅ Saved: International Routes Feb 2025.pdf (256,536 bytes)
✅ Saved: List of Major Air India Disasters _ Crashes, Death Toll, Tata Group, History, & Accidents _ Britannica.pdf (1,926,890 bytes)

📚 Total PDFs ready: 5
   • International Routes Feb 2025.pdf
   • Domestic Routes Feb 2025.pdf
   • Air India Fact She

## 🧠 Step 4: Build the Vector Store

This loads your PDFs, splits them into chunks, creates embeddings using
HuggingFace's `all-MiniLM-L6-v2` (free, runs locally), and stores them in ChromaDB.

In [6]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from uuid import uuid4

CHROMA_DIR = "/content/chroma_vectorstore"
COLLECTION_NAME = "air_india_docs"

print("📖 Loading PDF documents...")
loader = PyPDFDirectoryLoader(PDF_DIR, glob="**/*.pdf")
documents = loader.load()
print(f"   Loaded {len(documents)} page(s) from PDF(s)")

print("✂️  Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)
chunks = text_splitter.split_documents(documents)
print(f"   Created {len(chunks)} chunks")

print("🔢 Loading HuggingFace embedding model (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("   Embedding model ready!")

print("🗄️  Building ChromaDB vector store (this may take a few minutes)...")
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

uuids = [str(uuid4()) for _ in range(len(chunks))]
# Add in batches to avoid memory issues
BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch_docs = chunks[i:i+BATCH_SIZE]
    batch_ids = uuids[i:i+BATCH_SIZE]
    vector_store.add_documents(documents=batch_docs, ids=batch_ids)
    print(f"   Added batch {i//BATCH_SIZE + 1}/{(len(chunks)-1)//BATCH_SIZE + 1}")

print(f"\n✅ Vector store built with {len(chunks)} chunks and saved to {CHROMA_DIR}")

📖 Loading PDF documents...
   Loaded 121 page(s) from PDF(s)
✂️  Splitting documents into chunks...
   Created 243 chunks
🔢 Loading HuggingFace embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Embedding model ready!
🗄️  Building ChromaDB vector store (this may take a few minutes)...
   Added batch 1/3
   Added batch 2/3
   Added batch 3/3

✅ Vector store built with 243 chunks and saved to /content/chroma_vectorstore


## 🤖 Step 5: Set Up the LLM

Choose between **Groq (LLaMA Instant)** or **OpenAI (GPT-3.5-turbo)**.
Groq is recommended — it's fast and free.

In [7]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

# ─────────────────────────────────────────────────────────────
# 👇 Choose your LLM provider: "groq" or "openai"
LLM_PROVIDER = "groq"   # Change to "openai" if preferred
# ─────────────────────────────────────────────────────────────

if LLM_PROVIDER == "groq":
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is not set. Add it to Colab Secrets.")
    llm = ChatGroq(
        model="llama-3.1-8b-instant",   # Groq's LLaMA Instant model
        temperature=0,
        max_tokens=512,
        groq_api_key=GROQ_API_KEY,
    )
    print("✅ Using Groq — llama-3.1-8b-instant")

elif LLM_PROVIDER == "openai":
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is not set. Add it to Colab Secrets.")
    llm = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0,
        max_tokens=512,
        openai_api_key=OPENAI_API_KEY,
    )
    print("✅ Using OpenAI — gpt-3.5-turbo")

else:
    raise ValueError(f"Unknown LLM_PROVIDER: '{LLM_PROVIDER}'. Use 'groq' or 'openai'.")

✅ Using Groq — llama-3.1-8b-instant


## 🔗 Step 6: Build the RAG Chain (LangChain 0.2+ compatible)

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# ── Retriever ──────────────────────────────────────────────────
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# ── Prompt (LangChain 0.2+ style using ChatPromptTemplate) ─────
RAG_PROMPT_TEMPLATE = """\
You are a helpful and knowledgeable Air India assistant.
Use ONLY the context below to answer the user's question.
If the answer is not in the context, say: "I don't have enough information to answer that from the provided documents."
Keep your answer concise, accurate, and friendly.

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

# ── Helper: format retrieved docs into a single string ─────────
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}, Page: {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
        for doc in docs
    )

# ── RAG Chain (LCEL — LangChain Expression Language) ───────────
rag_chain = (
    RunnableParallel(
        context=(retriever | format_docs),
        question=RunnablePassthrough(),
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain is ready!")

✅ RAG chain is ready!


## 🧪 Step 7: Quick Test (Optional)

In [9]:
def get_response(question: str) -> dict:
    """Run a question through the RAG chain and return answer + source docs."""
    # Retrieve relevant docs for display
    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    sources = list({
        f"{doc.metadata.get('source', 'unknown')} (p.{doc.metadata.get('page', 'N/A')})"
        for doc in source_docs
    })
    return {"answer": answer, "sources": sources}

# Quick sanity check
test_q = "What is Air India?"
print(f"Q: {test_q}\n")
result = get_response(test_q)
print(f"A: {result['answer']}")
print(f"\n📎 Sources used:")
for s in result['sources']:
    print(f"   • {s}")

Q: What is Air India?

A: Air India is a full-service global airline and a major international carrier, founded by JRD Tata in 1932. It is a member of the Star Alliance and has been a part of the Tata group since January 2022.

📎 Sources used:
   • /content/air_india_docs/Air India Fact Sheet.pdf (p.1)
   • /content/air_india_docs/Aiesl Employees service regulation.pdf (p.0)


## 🚀 Step 8: Launch the Gradio Chat App

Run the cell below to launch a fully interactive chatbot UI with conversation history.

In [10]:
import gradio as gr

# ── Chat function with history support ────────────────────────
def chat(user_message: str, history: list):
    """Process a user message and return (answer, updated_history)."""
    if not user_message.strip():
        return "", history

    result = get_response(user_message)
    answer = result["answer"]

    # Append source footnote
    if result["sources"]:
        sources_str = "\n".join(f"• {s}" for s in result["sources"])
        answer += f"\n\n📎 **Sources:**\n{sources_str}"

    history.append((user_message, answer))
    return "", history

def clear_chat():
    return [], []

# ── Gradio UI ─────────────────────────────────────────────────
with gr.Blocks(
    title="✈️ Air India RAG Chatbot",
    theme=gr.themes.Soft(primary_hue="red"),
    css=".gradio-container { max-width: 860px; margin: auto; }"
) as demo:

    gr.HTML("""
        <div style="text-align:center; padding: 20px 0 10px;">
            <h1 style="font-size:2rem; color:#c8102e;">✈️ Air India RAG Chatbot</h1>
            <p style="color:#555;">Ask questions about Air India based on your uploaded documents.</p>
        </div>
    """)

    chatbot = gr.Chatbot(
        label="Chat",
        height=480,
        bubble_full_width=False,
        show_copy_button=True,
    )

    with gr.Row():
        txt = gr.Textbox(
            placeholder="Ask something about Air India... (press Enter or click Send)",
            show_label=False,
            scale=8,
        )
        send_btn = gr.Button("Send ✈️", variant="primary", scale=1)

    with gr.Row():
        clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

    # Sample questions
    gr.Examples(
        examples=[
            ["What are Air India's international routes?"],
            ["Tell me about Air India's history and ownership."],
            ["What domestic routes does Air India operate?"],
            ["What major accidents has Air India had?"],
            ["What are the employee service regulations?"],
        ],
        inputs=txt,
        label="💡 Sample Questions",
    )

    # State to hold conversation history
    state = gr.State([])

    # Event bindings
    txt.submit(chat, [txt, state], [txt, chatbot])
    txt.submit(lambda h: h, [chatbot], [state])

    send_btn.click(chat, [txt, state], [txt, chatbot])
    send_btn.click(lambda h: h, [chatbot], [state])

    clear_btn.click(clear_chat, outputs=[chatbot, state])

demo.launch(
    share=True,        # Creates a public link for Colab
    debug=False,
    quiet=True,
)

print("🎉 App launched! Click the public link above to open the chatbot.")

/tmp/ipykernel_1639/2277097647.py:24: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1639/2277097647.py:24: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1639/2277097647.py:37: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1639/2277097647.py:37: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1

* Running on public URL: https://e7c6d2b2bcaac54a68.gradio.live


🎉 App launched! Click the public link above to open the chatbot.


---
## ♻️ Optional: Reload Existing Vector Store (Skip PDF re-ingestion)

If you've already built the vector store in a previous session and it's still on disk,
run this cell instead of Step 4 to reload it quickly.

In [11]:
# ── Reload an existing ChromaDB vector store ───────────────────
# Only run this if you skipped Step 4 (already have the vector store)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = "/content/chroma_vectorstore"
COLLECTION_NAME = "air_india_docs"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

count = vector_store._collection.count()
print(f"✅ Reloaded vector store with {count} chunks from {CHROMA_DIR}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reloaded vector store with 486 chunks from /content/chroma_vectorstore


In [14]:
app_code = '''import os
import gradio as gr
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# ── Config ────────────────────────────────────────────────────
CHROMA_DIR = "./chroma_vectorstore"
COLLECTION_NAME = "air_india_docs"
LLM_PROVIDER = "groq"  # Change to "openai" if needed

# ── Embeddings & Vector Store ─────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# ── LLM ───────────────────────────────────────────────────────
if LLM_PROVIDER == "groq":
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=512)
else:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, max_tokens=512)

# ── RAG Chain ─────────────────────────────────────────────────
def format_docs(docs):
    return "\\n\\n".join(
        f"[Source: {doc.metadata.get(\'source\', \'unknown\')}, Page: {doc.metadata.get(\'page\', \'N/A\')}]\\n{doc.page_content}"
        for doc in docs
    )

prompt = ChatPromptTemplate.from_template("""
You are a helpful Air India assistant.
Use ONLY the context below to answer. If the answer is not in the context,
say: "I don\'t have enough information to answer that from the provided documents."

Context:
{context}

Question: {question}

Answer:""")

rag_chain = (
    RunnableParallel(context=(retriever | format_docs), question=RunnablePassthrough())
    | prompt | llm | StrOutputParser()
)

# ── Response helper ───────────────────────────────────────────
def get_response(question):
    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    sources = list({
        f"{doc.metadata.get(\'source\', \'unknown\')} (p.{doc.metadata.get(\'page\', \'N/A\')})"
        for doc in source_docs
    })
    return {"answer": answer, "sources": sources}

# ── Gradio UI ─────────────────────────────────────────────────
def chat(user_message, history):
    if not user_message.strip():
        return "", history
    result = get_response(user_message)
    answer = result["answer"]
    if result["sources"]:
        answer += "\\n\\n📎 **Sources:**\\n" + "\\n".join(f"• {s}" for s in result["sources"])
    history.append((user_message, answer))
    return "", history

def clear_chat():
    return [], []

with gr.Blocks(title="✈️ Air India RAG Chatbot", theme=gr.themes.Soft(primary_hue="red")) as demo:
    gr.HTML("""
        <div style="text-align:center; padding:20px 0 10px;">
            <h1 style="font-size:2rem; color:#c8102e;">✈️ Air India RAG Chatbot</h1>
            <p style="color:#555;">Ask questions about Air India based on your uploaded documents.</p>
        </div>
    """)
    chatbot = gr.Chatbot(label="Chat", height=480, bubble_full_width=False, show_copy_button=True)
    with gr.Row():
        txt = gr.Textbox(placeholder="Ask something about Air India...", show_label=False, scale=8)
        send_btn = gr.Button("Send ✈️", variant="primary", scale=1)
    clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")
    gr.Examples(
        examples=[
            ["What are Air India\'s international routes?"],
            ["Tell me about Air India\'s history and ownership."],
            ["What domestic routes does Air India operate?"],
        ],
        inputs=txt, label="💡 Sample Questions",
    )
    state = gr.State([])
    txt.submit(chat, [txt, state], [txt, chatbot])
    txt.submit(lambda h: h, [chatbot], [state])
    send_btn.click(chat, [txt, state], [txt, chatbot])
    send_btn.click(lambda h: h, [chatbot], [state])
    clear_btn.click(clear_chat, outputs=[chatbot, state])

if __name__ == "__main__":
    demo.launch()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py saved! Run it locally with: python app.py")

✅ app.py saved! Run it locally with: python app.py
